[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/40_linear_regression.ipynb)

# 🟡 Medium: Linear Regression

Реализуйте **linear regression** тремя способами в чистом PyTorch. Это упражнение связывает линейную алгебру, ручное gradient descent и стандартный training loop на одной задаче, поэтому результаты трёх методов должны быть близки.

Given data `X` of shape `(N, D)` and targets `y` of shape `(N,)`, find weight `w` of shape `(D,)` and bias `b` (scalar) such that:

$$\hat{y} = Xw + b$$

### Что оптимизируется
Обычно минимизируется mean squared error:

$$\mathrm{MSE}=\frac{1}{N}\sum_{i=1}^{N}(\hat y_i-y_i)^2$$

Weight `w` задаёт вклад каждой feature, bias `b` — общий сдвиг. Для хранения bias внутри общей матрицы можно добавить к `X` колонку единиц.

### Контракт
```python
class LinearRegression:
    def closed_form(self, X: Tensor, y: Tensor) -> tuple[Tensor, Tensor]: ...
    def gradient_descent(self, X: Tensor, y: Tensor, lr=0.01, steps=1000) -> tuple[Tensor, Tensor]: ...
    def nn_linear(self, X: Tensor, y: Tensor, lr=0.01, steps=1000) -> tuple[Tensor, Tensor]: ...
```

All methods return `(w, b)` where `w` has shape `(D,)` and `b` has shape `()`.

### Метод 1 — closed form
Дополните `X` колонкой единиц и решите least-squares system:

$$\theta = (X_{aug}^T X_{aug})^{-1} X_{aug}^T y$$

Предпочитайте `torch.linalg.lstsq` или решение системы явному вычислению inverse: это численно устойчивее и работает для переопределённой задачи. Из solution отделите первые `D` значений как weight и последнее как scalar bias.

### Метод 2 — gradient descent вручную
Инициализируйте `w` и `b` нулями. На каждой итерации вычислите predictions, errors и аналитические gradients MSE:
```
pred = X @ w + b
error = pred - y
grad_w = (2/N) * X^T @ error
grad_b = (2/N) * error.sum()
w -= lr * grad_w
b -= lr * grad_b
```

Следите за shape: `grad_w` должен быть `(D,)`, `grad_b` — scalar. Обновления выполняются без autograd по условию упражнения. Слишком большой learning rate вызывает расходимость, слишком малый — медленную convergence.

### Метод 3 — `nn.Linear` и autograd
Создайте `nn.Linear(D,1)`, используйте `nn.MSELoss`, optimizer и обычный цикл `zero_grad → forward → loss → backward → step`. Output слоя имеет `(N,1)`, тогда как target `(N,)`, поэтому выровняйте shapes и не допускайте случайного broadcasting в MSE. После обучения извлеките weight как `(D,)`, bias как scalar.

### Ограничения
- все входы и результаты — PyTorch tensors;
- не используйте NumPy или sklearn;
- closed form не использует итеративную optimization;
- gradient descent вычисляет gradients вручную без autograd;
- `nn_linear` использует `nn.Linear` и `loss.backward()`.

### Быстрые самопроверки
- на noiseless synthetic data параметры восстанавливаются близко к истинным;
- `w.shape == (D,)`, `b.shape == ()`;
- MSE уменьшается во время gradient descent;
- три метода дают близкие predictions и параметры;
- closed-form output не требует gradient.

### Типичные ошибки
- явный inverse плохо обусловленной `X.T @ X`;
- забытая колонка единиц и потерянный bias;
- коэффициент `2/N` отсутствует в ручном gradient;
- broadcasting `(N,1)` и `(N,)` создаёт loss формы `(N,N)`;
- gradient не обнуляется в `nn.Linear` loop;
- параметры возвращаются с лишними dimensions.

### Официальные материалы и примеры
- [`torch.linalg.lstsq`](https://docs.pytorch.org/docs/stable/generated/torch.linalg.lstsq.html) — устойчивое least-squares solution и примеры;
- [`torch.nn.Linear`](https://docs.pytorch.org/docs/stable/generated/torch.nn.Linear.html) — trainable linear transformation;
- [Optimizing Model Parameters](https://docs.pytorch.org/tutorials/beginner/basics/optimization_tutorial.html) — полный training loop с optimizer и loss.

### Вопросы для самопроверки
1. Зачем к `X` добавляют колонку единиц?
2. Почему `lstsq` предпочтительнее явного inverse?
3. Как убедиться, что broadcasting не изменил смысл MSE?

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch
import torch.nn as nn

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

class LinearRegression:
    def closed_form(self, X: torch.Tensor, y: torch.Tensor):
        """Normal equation: w = (X^T X)^{-1} X^T y"""
        pass  # Return (w, b)

    def gradient_descent(self, X: torch.Tensor, y: torch.Tensor,
                         lr: float = 0.01, steps: int = 1000):
        """Manual gradient descent loop"""
        pass  # Return (w, b)

    def nn_linear(self, X: torch.Tensor, y: torch.Tensor,
                  lr: float = 0.01, steps: int = 1000):
        """Train nn.Linear with autograd"""
        pass  # Return (w, b)

In [ ]:
# 🧪 Debug
torch.manual_seed(42)
X = torch.randn(100, 3)
true_w = torch.tensor([2.0, -1.0, 0.5])
y = X @ true_w + 3.0

model = LinearRegression()

w_cf, b_cf = model.closed_form(X, y)
print(f"Closed-form:  w={w_cf}, b={b_cf.item():.4f}")

w_gd, b_gd = model.gradient_descent(X, y, lr=0.05, steps=2000)
print(f"Grad descent: w={w_gd}, b={b_gd.item():.4f}")

w_nn, b_nn = model.nn_linear(X, y, lr=0.05, steps=2000)
print(f"nn.Linear:    w={w_nn}, b={b_nn.item():.4f}")

print(f"\nTrue:         w={true_w}, b=3.0")

In [ ]:
# ✅ SUBMIT
from torch_judge import check
check("linear_regression")